#About

This notebook contains code for fine-tuning gemma3-4b model for text summarization using custom dataset.

Quantized 4bit model from Unsloth is used for fine-tuning.

Fine-tuning is done on texts shorter than 16k characters. Evaluation (separate notebook) is done on the longer texts. This is due to computational limitations.



#Setup

In [ ]:
%autosave 60

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
hf_token = "<hf_token>"

from huggingface_hub import login
login(token=hf_token)

In [ ]:
import wandb

In [ ]:
wandb.login(key="<wandb_token>")

In [ ]:
%%capture
import os, re
# Do this only in Colab notebooks! Otherwise use pip install unsloth
import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
DATA_PATH = "drive/MyDrive/summaries-data/LITEKO_LABELED_DATASET"
MODEL_SAVE_PATH = "drive/MyDrive/fine-tuned-summaries/gemma3-4b"

In [ ]:
SYSTEM_PROMPT = """You are a lawyer.
You will be given a text and you will summarize it.
The summary MUST be in lithuanian.
The summary MUST be a continuous text without any formatting.
The summary MIGHT have multiple paragraphs."""

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import pandas as pd
from peft import LoraConfig, get_peft_model
from unsloth import FastLanguageModel
from datasets import Dataset, load_from_disk, concatenate_datasets
import numpy as np
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import train_on_responses_only

#Load data

In [ ]:
dataset = load_from_disk(DATA_PATH)
dataset

In [ ]:
dataset[0]

In [ ]:
train_dataset = dataset.filter(lambda row: len(row['Article']) <= 16000)
train_dataset

In [ ]:
train_dataset = train_dataset.map(
    lambda example: {
        "article_length": len(example["Article"]),
        "summary_length": len(example["Summary"])
    }
)
train_dataset

In [ ]:
article_lengths = train_dataset["article_length"]
summary_lengths = train_dataset["summary_length"]

percentiles = [0, 25, 50, 75, 90, 100]

article_percentiles = np.percentile(article_lengths, percentiles)
summary_percentiles = np.percentile(summary_lengths, percentiles)

for p, a, s in zip(percentiles, article_percentiles, summary_percentiles):
    print(f"{p}th percentile - Article: {a:.1f}, Summary: {s:.1f}")

#Load model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    max_seq_length = 10000,
    dtype = torch.bfloat16,
    load_in_4bit = True,
)

In [ ]:
print(next(model.parameters()).dtype)
print(model.device)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 123,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
model.print_trainable_parameters()

#Prepare data for training

In [ ]:
def generate_conversation(examples):
    problems  = examples["Article"]
    solutions = examples["Summary"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "system", "content" : SYSTEM_PROMPT},
            {"role" : "user", "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return {"conversations": conversations}

In [ ]:
train_dataset = train_dataset.map(generate_conversation, batched=True)
train_dataset

In [ ]:
train_dataset["conversations"][0]

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

In [ ]:
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
train_dataset

In [ ]:
train_dataset["text"][0]

In [ ]:
def calculate_text_length(example):
    example['text_length'] = tokenizer(example['text'], return_tensors="pt")['input_ids'].shape[1]
    return example

train_dataset = train_dataset.map(calculate_text_length, batched=False)
train_dataset

In [ ]:
text_lengths = train_dataset["text_length"]

percentiles = [0, 25, 50, 75, 90, 100]

text_percentiles = np.percentile(text_lengths, percentiles)

for p, a in zip(percentiles, text_percentiles):
    print(f"{p}th percentile - Article: {a:.1f}")

In [ ]:
split = train_dataset.train_test_split(test_size=0.2)
split

#Train model

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split["train"],
    eval_dataset = split["test"],
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        logging_strategy="epoch",
        eval_strategy="epoch",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 123,
        output_dir = "outputs",
        report_to = "wandb",
    ),
)

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
trainer_stats = trainer.train()

#Save model

In [ ]:
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)